# Audit penyimpanan encoder — read-only

Pengemasan multi-function berhasil menjaga graph, bobot, kontrak, serta output CPU pada kasus yang diuji. Namun dua encoder hanya turun dari 1.103.400.719 ke 1.058.208.222 byte raw: hemat **43,10 MiB / 4,10%**. Paket compiled sekitar 1.009 MiB untuk encoder saja. Paket ini belum dipasang ke aplikasi; belum ada pengukuran kecepatan iPhone untuk format gabungan.

Audit ini membaca paket yang sama dan **tidak menulis model**. Ia membandingkan konstanta yang identik bitwise, metadata pengelompokan bobot, hasil decode LUT yang identik walau encoding berbeda, dan nilai flatten identik pada layout berbeda. Hash payload model yang berbeda sendiri belum menunjukkan berapa banyak bobot berbeda. Jangan menyimpulkan bahwa seluruh bobot berbeda atau bahwa encoding palette pasti penyebabnya sebelum audit selesai.

Output hanya JSON berisi hash, shape dan statistik, tanpa menyalin audio atau bobot. MusicFM dan MuQ diproses terpisah. Tidak melakukan inferensi audio, training, konversi, kuantisasi, atau pemasangan model. Beberapa bobot di-decode pada CPU untuk perbandingan bitwise; proses ini memakai memori sementara.


In [ ]:
from pathlib import Path
from datetime import datetime
import json,hashlib,subprocess
ROOT=Path.cwd().resolve()
assert (ROOT/'scripts/audit_encoder_sharing.py').is_file(), 'Buka notebook dari root SongMap'
RUN_PYTHON=ROOT.parent/'Section-Annotations/.venv-songformer311/bin/python'
assert RUN_PYTHON.is_file(), f'Python venv tidak ditemukan: {RUN_PYTHON}'
def sha(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED={'musicfm':'1f5abdd9aa4fa4d3dec4cf0cc038a203a4aae8e914218876a270c668da500f5d',
          'muq':'3c5a5bf912f393cbdef908af69642f9899ea06f06a4bcfba18b44770e39f8149'}
SOURCES={}
for role,expected in EXPECTED.items():
    paths=(ROOT/'outputs.local').glob('encoder-weight-packaging-*/'+role+'-*/packaging-report.json')
    SOURCES[role]=next((p.resolve() for p in paths if sha(p)==expected),None)
    assert SOURCES[role], f'Report {role} yang sudah ditinjau belum ditemukan'
OUTPUT=ROOT/'outputs.local'/('encoder-sharing-audit-'+datetime.now().strftime('%Y%m%d-%H%M%S-%f'))
print('Input:',SOURCES)
print('Output JSON:',OUTPUT)

def audit(role):
    output=OUTPUT/(role+'-audit.json')
    if output.is_file():
        r=json.loads(output.read_text())
        assert r['packaging_report_sha256']==EXPECTED[role] and r['audit_script_sha256']==sha(ROOT/'scripts/audit_encoder_sharing.py')
        print('Audit selesai digunakan kembali:',output);return
    log=OUTPUT/(role+'.log');log.parent.mkdir(parents=True,exist_ok=True)
    args=[RUN_PYTHON,'-u',ROOT/'scripts/audit_encoder_sharing.py','--packaging-report',SOURCES[role],'--output',output]
    with log.open('w',encoding='utf-8') as handle:
        process=subprocess.Popen([str(x) for x in args],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='');handle.write(line);handle.flush()
            status=process.wait()
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    if status:raise RuntimeError(f'Audit gagal ({status}); lihat {log}. Tidak mengubah model sumber.')


In [ ]:
audit('musicfm')


In [ ]:
audit('muq')


In [ ]:
from IPython.display import display,Markdown
lines=['| Encoder | Blob bobot packed MiB | File lain MiB | Grup konstanta identik | Grup decoded sama, encoding berbeda | Grup layout berbeda |',
       '|---|---:|---:|---:|---:|---:|']
for role in ('musicfm','muq'):
    path=OUTPUT/(role+'-audit.json');r=json.loads(path.read_text())
    lines.append(f"| {role} | {r['packed_weight_blob_bytes']/2**20:.1f} | {r['packed_other_bytes']/2**20:.1f} | {len(r['exact_constant_groups'])} | {len(r['decoded_identical_different_encoding_groups'])} | {len(r['decoded_flat_identical_different_shape_groups'])} |")
    print(role,'report:',path)
display(Markdown('\n'.join(lines)))
print('Kirim musicfm-audit.json dan muq-audit.json.')
print('Estimasi kategori saling tumpang tindih, mencakup data yang mungkin sudah dibagi, dan bukan tambahan penghematan ukuran paket.')


Dari hasil ini kita bisa memilih langkah pengemasan berikutnya berdasarkan bukti. Konstanta dengan nilai identik tetapi metadata berbeda, encoding LUT berbeda dengan hasil decode identik, atau layout berbeda memerlukan penanganan yang berbeda. Jika nilai hasil decode berbeda, menyatukannya akan mengubah bobot dan memerlukan evaluasi kualitas; audit ini tidak melakukannya.

Tidak ada perubahan route CPU/GPU, model, boundary, anotasi, atau checkpoint pada tahap ini. Kandidat runtime tetap frontend CPU dan encoder window penuh CPU+GPU yang sudah diuji.

Audit model asli yang ditinjau tidak menemukan kandidat encoding/layout yang decode-nya sama; konstanta identik hampir seluruhnya sudah dibagi. Lanjutkan [encoder_canonical_bank.ipynb](encoder_canonical_bank.ipynb) untuk kandidat satu bank yang mengubah bobot short/partial dan memerlukan evaluasi kualitas baru.
